In [1]:
// Charger le pont ciblé Stim et fixer la culture des sorties numériques.
#r "nuget: pythonnet,3.1.0"
#load "../../Config/PythonNetLoader.cs"
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;
using System.Numerics;
using System.Text.Json;
using System.Globalization;
using Python.Runtime;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;
Console.WriteLine("CHP et état complet natifs C# ; Stim par Python.NET (version affichée au chargement).");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages pythonnet, 3.1.0

CHP et état complet natifs C# ; Stim par Python.NET (version affichée au chargement).


# Complexity-06b — Déquantifier les suprématies — C# (Aaronson–Gottesman)

**Navigation** : [Index](README.md) | [06 — Simuler un circuit quantique classiquement](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) | [06c — Classical at heart →](Complexity-06c-ClassicalAtHeart-Python.ipynb)

Gottesman–Knill comme arbitre exécutable (Aaronson–Gottesman, *Improved Simulation of Stabilizer Circuits*, 2004). Cette accrétion de Complexity-06 (à venir) tient la simulation classique polynomiale des circuits de Clifford, selon leur longueur et le coût de chaque opération.

**Jumeau C#.** CHP, règles de Pauli et état complet sont natifs ; Stim est appelé réellement par Python.NET comme troisième juge. Installer `stim==1.15.0` dans le Python choisi par `Config/PythonNetLoader.cs`. Les générateurs aléatoires diffèrent entre langages ; les budgets et distributions sont conservés, les temps ne sont pas comparés entre langages. [Source Python](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb).


## 0. Pourquoi ce notebook

Ce notebook est le sixième de la série
([01 — Compter des pas](Complexity-01-StepCounting-Python.ipynb),
[02 — Vérifier ou trouver](Complexity-02-P-NP-Reduction-Python.ipynb),
[03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb),
[03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb),
[04b — conjectures online](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb),
[05b — Aaronson–Arkhipov](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb)) :
un texte fondateur, **fait tourner**. Le geste propre à cette veine est que la thèse
**s'exécute**. Quand une « suprématie quantique » est annoncée, la question qui installe
ou réfute la claim n'est pas « est-ce quantique ? » mais « **dans quelle classe de
circuits ?** ». Gottesman–Knill (1998) : tout circuit de portes de Clifford
($H$, $S$, $\mathsf{CNOT}$, mesures) se simule en temps **polynomial classique** —
l'avantage quantique ne peut pas vivre là. Aaronson–Gottesman (2004) en donnent l'organe :
le port C# de l'algorithme CHP sur un tableau de bits, sans jamais écrire une amplitude. Ici les portes
coûtent $O(n)$ chacune ; la lecture déterministe par élimination GF(2) peut coûter
$O(n^3)$ par mesure. Le coût total dépend aussi de la longueur du circuit. Nous allons donc :

1. **exécuter le formalisme stabilisateur** : le tableau de Pauli d'Aaronson–Gottesman,
   avec sa table de phase et ses règles de conjugaison **dérivées des matrices
   $2\times2$** — jamais recopiées d'une table magique ;
2. **construire les deux machines** : l'arbitre polynomial (port CHP) et l'état complet
   $2^n$, puis les **croiser** sur des circuits Clifford aléatoires — les valeurs
   propres du groupe stabilisateur, calculées indépendamment par les deux machines,
   doivent coïncider **exactement** ; le simulateur de référence du métier
   ([stim](https://github.com/quantumlib/Stim), Gidney 2021) sert de troisième juge
   quand il est installé ;
3. **mesurer le régime, pas l'objet** : à circuits Clifford purs, l'écart de coût entre
   l'état complet et l'arbitre varie avec la taille ; une **seule** porte $T$ ferme
   l'arbitre — et ferme aussi stim. La suprématie est une assertion sur un *régime*,
   pas sur un objet : c'est la leçon de méthode que ce banc rend mesurable.

L'encart de cadrage, sans mesure : les épisodes contemporains de réfutation de claims
de suprématie (RCS, GBS) relèvent ici exactement de la **leçon de méthode** — comment on
réfute proprement, en exhibant le régime où la claim casse. Ce notebook ne prend parti
pour aucun verdict : il mesure deux régimes sur des instances bornées, et chacun peut
réexécuter le banc.

## 0.1. Prérequis quantiques : qubit, ket, Bell, GHZ

Le carnet utilise la **notation de Dirac** ($|\psi\rangle$) sans la présenter :
toute la grammaire ket est supposée acquise, et la promesse polynomialiste de
Gottesman–Knill s'énonce sans elle. Cet encart pose les cinq objets que la suite
emploie sans redéfinition, et indique ce qu'aucun d'eux n'est.

**1. Qubit.** Un qubit est un espace de Hilbert de dimension 2 sur $\mathbb{C}$.
La base **orthonormée canonique** (dite « base computationnelle ») est

$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix},\qquad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}.$$

Tout qubit est une combinaison linéaire $a|0\rangle + b|1\rangle$ avec
$|a|^2 + |b|^2 = 1$ (vecteur unitaire). La mesure dans la base $\{|0\rangle,|1\rangle\}$
donne $0$ avec probabilité $|a|^2$ et $1$ avec probabilité $|b|^2$.

**2. États à un qubit.** Deux superpositions canoniques reviennent partout :

$$|+\rangle = \tfrac{1}{\sqrt{2}}\bigl(|0\rangle + |1\rangle\bigr),\qquad |-\rangle = \tfrac{1}{\sqrt{2}}\bigl(|0\rangle - |1\rangle\bigr).$$

$|+\rangle$ est l'état propre $+1$ de $X$ (la porte de Hadamard $H$ envoie
$|0\rangle$ sur $|+\rangle$).

**3. Produit tensoriel.** $n$ qubits vivent dans l'espace de Hilbert
$(\mathbb{C}^2)^{\otimes n}$, de dimension $2^n$. La base est
$\{|x_0 x_1 \ldots x_{n-1}\rangle\}$ pour $x \in \{0,1\}^n$. Le carnet
emploie la convention « qubit 0 = bit de poids fort » (l'ordre qui rend la
construction de Bell et GHZ lisible).

**4. États de Bell.** Le couple intriqué maximal à 2 qubits admet quatre
vecteurs de Bell, dont le plus employé :

$$|\Phi^+\rangle = \tfrac{1}{\sqrt{2}}\bigl(|00\rangle + |11\rangle\bigr).$$

$|\Phi^+\rangle$ est l'unique vecteur propre $+1$ du groupe engendré par
$X \otimes X$ et $Z \otimes Z$ — c'est exactement ce que la cellule de
démos (cellule 13) lira sur le tableau CHP. Les deux mesures de $Z$ sur les
qubits d'un état de Bell donnent des résultats **toujours égaux** : c'est la
corrélation EPR, et c'est la **seule** signature mesurable d'intrication que
ce carnet manipule.

**5. État GHZ-$n$.** Pour $n$ qubits, l'analogue multi-qubits est

$$|\mathrm{GHZ}_n\rangle = \tfrac{1}{\sqrt{2}}\bigl(|0\rangle^{\otimes n} + |1\rangle^{\otimes n}\bigr).$$

Le groupe stabilisateur de $|\mathrm{GHZ}_n\rangle$ est engendré par
$X_0 X_1 \ldots X_{n-1}$ (le produit des résultats des $n$ mesures $X$ vaut $+1$) et $Z_0 Z_j$ pour $j=1,\ldots,n-1$ (toutes les
mesures $Z$ sont égales entre elles). Un résultat individuel $X_0$ n’a pas de
relation déterministe avec ces résultats $Z$. Le carnet
calcule ce stabilisateur sur $n=8$ à la cellule 13 — le coût pour l'arbitre
CHP est le même que pour un produit tensoriel $|0\rangle^{\otimes 8}$, ce
qui est la **promesse polynomialiste** que toute la suite mesure.

**Ce que cet encart n'est pas.** Ni une introduction à la mécanique quantique,
ni un rappel d'algèbre linéaire : le formalisme stabilisateur, la table de
phase de Pauli et les règles de conjugaison de $H$, $S$, $\mathsf{CNOT}$ sont
construits et prouvés dans les cellules qui suivent (cette section n'est pas
un raccourci vers elles). Les prérequis sont uniquement **notationnels** :
savoir lire $|\psi\rangle$, savoir qu'un produit tensoriel a $2^n$ composantes,
savoir que Bell et GHZ sont des vecteurs de $\mathbb{C}^{2^n}$ avec une
description en deux mots. Le reste est l'objet du carnet.

In [2]:
Console.WriteLine("Seeds : 0, 1, 7 ; durées en secondes, médiane et enveloppe.");


Seeds : 0, 1, 7 ; durées en secondes, médiane et enveloppe.


## 1. Le formalisme stabilisateur, définition exécutée

Un état stabilisateur est l'unique vecteur propre $+1$ commun de $n$ chaînes de Pauli
indépendantes qui commutent — ses **stabilisateurs**. Aaronson–Gottesman stockent le
groupe étendu : $n$ **déstabilisateurs** (les partenaires qui complètent la base
symplectique) plus $n$ stabilisateurs, chacun encodé en bits par qubit —
$I=(0,0)$, $X=(1,0)$, $Z=(0,1)$, $Y=(1,1)$, le $Y$ portant les deux bits puisque
$Y = iXZ$ — plus un bit de signe $r$ par ligne et une ligne de travail.

Tout ce qui suit manipule des **lignes de bits** : aucune amplitude, aucun nombre
complexe hors les matrices $2\times2$ qui servent à *prouver* les tables. C'est la
promesse polynomialiste de Gottesman–Knill, rendue littérale.

In [3]:
// Les produits matriciels dérivent la table de phase sans recopier une table externe.
Complex[, ] Mul(Complex[, ] a, Complex[, ] b)
{
    int n = a.GetLength(0), m = b.GetLength(1), p = a.GetLength(1);
    var c = new Complex[n, m];
    for (int i = 0; i < n; i++)
        for (int j = 0; j < m; j++)
            for (int k = 0; k < p; k++)
                c[i, j] += a[i, k] * b[k, j];
    return c;
}

// Le produit tensoriel construit les opérateurs agissant sur deux qubits.
Complex[, ] Kronecker(Complex[, ] a, Complex[, ] b)
{
    int n = a.GetLength(0), m = a.GetLength(1), p = b.GetLength(0), q = b.GetLength(1);
    var c = new Complex[n * p, m * q];
    for (int i = 0; i < n; i++)
        for (int j = 0; j < m; j++)
            for (int k = 0; k < p; k++)
                for (int l = 0; l < q; l++)
                    c[i * p + k, j * q + l] = a[i, j] * b[k, l];
    return c;
}

bool EqualMatrix(Complex[, ] a, Complex[, ] b, Complex factor)
{
    for (int i = 0; i < a.GetLength(0); i++)
        for (int j = 0; j < a.GetLength(1); j++)
            if ((a[i, j] - factor * b[i, j]).Magnitude > 1e-10)
                return false;
    return true;
}

var I = Complex.ImaginaryOne;
Complex[][, ] pauli =
{
    new Complex[, ]
    {
        {
            1,
            0
        },
        {
            0,
            1
        }
    },
    new Complex[, ]
    {
        {
            0,
            1
        },
        {
            1,
            0
        }
    },
    new Complex[, ]
    {
        {
            1,
            0
        },
        {
            0,
            -1
        }
    },
    new Complex[, ]
    {
        {
            0,
            -I
        },
        {
            I,
            0
        }
    }
};
string[] lettres =
{
    "I",
    "X",
    "Z",
    "Y"
};
// Chaque exposant k encode la phase i^k du produit de deux Pauli.
int[, ] phases = new int[4, 4];
for (int a = 0; a < 4; a++)
    for (int b = 0; b < 4; b++)
    {
        // Identifier le Pauli et sa phase par comparaison matricielle.
        var produit = Mul(pauli[a], pauli[b]);
        bool trouve = false;
        for (int k = 0; k < 4; k++)
            for (int c = 0; c < 4; c++)
                if (EqualMatrix(produit, pauli[c], Complex.Pow(I, k)))
                {
                    phases[a, b] = k;
                    trouve = true;
                }

        if (!trouve)
            throw new Exception("Produit de Pauli introuvable");
    }

Console.WriteLine("Phases (I, X, Z, Y), dérivées des matrices :");
for (int a = 0; a < 4; a++)
    Console.WriteLine(string.Join(" ", Enumerable.Range(0, 4).Select(b => phases[a, b])));
var cx = new Complex[, ]
{
    {
        1,
        0,
        0,
        0
    },
    {
        0,
        1,
        0,
        0
    },
    {
        0,
        0,
        0,
        1
    },
    {
        0,
        0,
        1,
        0
    }
};
int nbRegles = 0, nbNegatives = 0;
// Les règles CNOT gardent le signe et les deux facteurs Pauli résultants.
var regles = new Dictionary<(int, int), (int signe, int premier, int second)>();
for (int a = 0; a < 4; a++)
    for (int b = 0; b < 4; b++)
    {
        // La conjugaison matricielle fournit un témoin indépendant des règles du tableau.
        var resultat = Mul(Mul(cx, Kronecker(pauli[a], pauli[b])), cx);
        bool trouve = false;
        for (int c = 0; c < 4; c++)
            for (int d = 0; d < 4; d++)
                foreach (int signe in new[]
                {
                    1,
                    -1
                }

                )
                    if (EqualMatrix(resultat, Kronecker(pauli[c], pauli[d]), signe))
                    {
                        Console.WriteLine($"{lettres[a]}{lettres[b]} -> {(signe > 0 ? "+" : "-")}{lettres[c]}{lettres[d]}");
                        nbRegles++;
                        if (signe < 0)
                            nbNegatives++;
                        regles[(a, b)] = (signe, c, d);
                        trouve = true;
                    }

        if (!trouve)
            throw new Exception("Conjugaison introuvable");
    }

if (nbRegles != 16 || nbNegatives != 2)
    throw new Exception("Conjugaisons CNOT incomplètes");


Phases (I, X, Z, Y), dérivées des matrices :


0 0 0 0


0 0 3 1


0 1 0 3


0 3 1 0


II -> +II


... (11 outputs left out)

YI -> +YX


YX -> +YI


YZ -> +XY


YY -> -XZ


**Lecture.** La table n'est pas symétrique : $X \cdot Y = iZ$ mais $Y \cdot X = -iZ$.
Suivre le cycle $X \to Y \to Z \to X$ coûte un facteur $i$ ($e = 1$) ; le remonter en
rend un ($e = 3$). Cette asymétrie est tout ce que la mécanique quantique ajoute au
calcul sur bits : le simulateur Clifford la trackera en **un seul bit par ligne**
($\sum_j e_j / 2 \bmod 2$), pas en amplitudes complexes.

## 2. Les règles de conjugaison, dérivées elles aussi

La second endroit où se tromper en silence : l'action des portes sur les lignes du
tableau. $H$ conjugue $X \leftrightarrow Z$ (et change le signe d'un $Y$), $S$
conjugue $X \to Y$ (et change le signe d'un $Y$), et le $\mathsf{CNOT}$ suit deux
xors — plus **un piège de phase** que l'on dérive au lieu de le croire : la table
complète des 16 conjugués $\mathsf{CNOT}\,P\,\mathsf{CNOT}^\dagger$ sur la paire
(contrôle, cible), calculée matriciellement.

In [4]:
Console.WriteLine($"CNOT : {nbRegles} règles, dont {nbNegatives} signes négatifs.");


CNOT : 16 règles, dont 2 signes négatifs.


**Lecture.** Les deux xors ($x_t \leftarrow x_t \oplus x_c$ puis
$z_c \leftarrow z_c \oplus z_t$) reproduisent les **lettres** des 16 conjugués. Le
signe $-$ n'apparaît que pour $X_c Z_t$ et $Y_c Y_t$ — soit, en bits, exactement
$x_c = 1$, $z_t = 1$ et $z_c = x_t$ (les bits lus *avant* les xors). Tout le reste du
notebook applique ces règles prouvées : la seule constante de phase du formalisme est
dérivée, aucune n'est recopiée.

## 3. L'organe : le simulateur CHP, complet

Trois pièces dans l'algorithme d'Aaronson–Gottesman :

1. **`RowSum(h, i)`** — additionner la ligne $i$ dans la ligne $h$ : Pauli xorés,
   phase accumulée par la table dérivée ci-dessus,
   $r_h \leftarrow r_i \oplus r_h \oplus (\tfrac{1}{2}\sum_j e_j \bmod 2)$. Pour
   deux lignes qui commutent (toujours le cas dans le groupe stabilisateur), la somme
   $\sum e_j$ est paire — le simulateur l'exige.
2. **Les portes** — règles de conjugaison dérivées du §2, $O(n)$ par porte. La porte
   $T = \mathrm{diag}(1, e^{i\pi/4})$ est **refusée** : elle n'est pas Clifford, le
   API stabilisatrice refuse cette porte hors Clifford. Ce refus ne prouve pas que
   chaque état après $T$ soit non stabilisateur (par exemple $T|0\rangle=|0\rangle$).
3. **La mesure $Z_q$** — *déterministe* ssi aucun stabilisateur n'anti-commute avec
   $Z_q$ : l'état est déjà propre, l'outcome se **lit** (résolution de $(0 | e_q)$
   dans le span GF(2) des stabilisateurs, produit par rowsum successifs — lecture pure,
   le tableau n'est pas modifié). Sinon *aléatoire* : outcome uniforme, et l'état
   post-mesure est stabilisé par $(-1)^m Z_q$. Cette implémentation utilise une
   élimination GF(2) pouvant coûter $O(n^3)$ par mesure déterministe ; la mesure
   aléatoire et la mise à jour du tableau sont au plus quadratiques.

Et en regard, la machine témoin : l'**état complet** $2^n$ — pas de classe de circuits,
il paie l'exponentielle partout, mais il simule tout, $T$ comprise. Le croisement du §4
dirige ces deux machines l'une contre l'autre.

In [5]:
// Les lignes 0..n-1 déstabilisent les stabilisateurs n..2n-1 ; la dernière est temporaire.
class CHP
{
    public readonly int N;
    public readonly int[, ] X, Z;
    public readonly int[] R;
    readonly Random rng;
    readonly int[, ] phase;
    public CHP(int n, int[, ] table, int seed = 0)
    {
        N = n;
        phase = table;
        rng = new Random(seed);
        // Les deux tableaux binaires codent les composantes X et Z de chaque Pauli.
        X = new int[2 * n + 1, n];
        Z = new int[2 * n + 1, n];
        R = new int[2 * n + 1];
        for (int j = 0; j < n; j++)
        {
            // L’état initial |0...0> est stabilisé par les Z individuels.
            X[j, j] = 1;
            Z[n + j, j] = 1;
        }
    }

    // Multiplie des lignes commutantes en tenant compte des phases de Pauli.
    public void RowSum(int h, int i)
    {
        // e accumule les exposants de phase avant réduction modulo quatre.
        int e = 0;
        for (int j = 0; j < N; j++)
            e += phase[X[i, j] + 2 * Z[i, j], X[h, j] + 2 * Z[h, j]];
        if (e % 2 != 0)
            throw new Exception("Lignes non commutantes");
        // Pour deux lignes commutantes la phase globale est réelle : seul son signe reste.
        R[h] ^= R[i] ^ ((e / 2) & 1);
        for (int j = 0; j < N; j++)
        {
            X[h, j] ^= X[i, j];
            Z[h, j] ^= Z[i, j];
        }
    }

    // H échange X et Z ; Y acquiert un signe négatif.
    public void H(int q)
    {
        for (int i = 0; i < 2 * N + 1; i++)
        {
            int x = X[i, q], z = Z[i, q];
            R[i] ^= x & z;
            X[i, q] = z;
            Z[i, q] = x;
        }
    }

    // S envoie X vers Y et actualise le signe quand la ligne contient déjà Z.
    public void S(int q)
    {
        for (int i = 0; i < 2 * N + 1; i++)
        {
            int x = X[i, q], z = Z[i, q];
            R[i] ^= x & z;
            Z[i, q] = z ^ x;
        }
    }

    // Les Pauli X et Z changent uniquement le signe des lignes anticommutantes.
    public void XGate(int q)
    {
        for (int i = 0; i < 2 * N + 1; i++)
            R[i] ^= Z[i, q];
    }

    public void ZGate(int q)
    {
        for (int i = 0; i < 2 * N + 1; i++)
            R[i] ^= X[i, q];
    }

    // La conjugaison propage X du contrôle vers la cible et Z en sens inverse.
    public void Cnot(int c, int t)
    {
        for (int i = 0; i < 2 * N + 1; i++)
        {
            int xc = X[i, c], zc = Z[i, c], xt = X[i, t], zt = Z[i, t];
            // Cette condition garde la phase produite par la conjugaison CNOT.
            if (xc == 1 && zt == 1 && zc == xt)
                R[i] ^= 1;
            X[i, t] = xt ^ xc;
            Z[i, c] = zc ^ zt;
        }
    }

    void CopyRow(int to, int from)
    {
        for (int j = 0; j < N; j++)
        {
            X[to, j] = X[from, j];
            Z[to, j] = Z[from, j];
        }

        R[to] = R[from];
    }

    void ClearRow(int i)
    {
        for (int j = 0; j < N; j++)
        {
            X[i, j] = 0;
            Z[i, j] = 0;
        }

        R[i] = 0;
    }

    // Élimination GF(2) avec coefficients pour exprimer Zq par les stabilisateurs.
    int Deterministe(int q)
    {
        // Les deux premiers blocs représentent X et Z ; le dernier suit les coefficients.
        int[, ] a = new int[N, 3 * N];
        bool[] used = new bool[N];
        for (int i = 0; i < N; i++)
        {
            for (int j = 0; j < N; j++)
            {
                a[i, j] = X[N + i, j];
                a[i, N + j] = Z[N + i, j];
            }

            // L’identité conserve la combinaison de générateurs au fil de l’élimination.
            a[i, 2 * N + i] = 1;
        }

        for (int col = 0; col < 2 * N; col++)
        {
            int pivot = -1;
            for (int i = 0; i < N; i++)
                if (!used[i] && a[i, col] != 0)
                {
                    pivot = i;
                    break;
                }

            if (pivot < 0)
                continue;
            used[pivot] = true;
            for (int i = 0; i < N; i++)
                if (i != pivot && a[i, col] != 0)
                    // XOR réalise l’élimination sur GF(2), coefficients compris.
                    for (int k = 0; k < 3 * N; k++)
                        a[i, k] ^= a[pivot, k];
        }

        int ligne = -1;
        for (int i = 0; i < N; i++)
        {
            bool egale = true;
            for (int j = 0; j < 2 * N; j++)
                if (a[i, j] != (j == N + q ? 1 : 0))
                {
                    egale = false;
                    break;
                }

            if (egale)
            {
                ligne = i;
                break;
            }
        }

        if (ligne < 0)
            throw new Exception("Z déterministe absent du groupe");
        // Reconstituer le produit signé permet de retrouver la valeur propre de Zq.
        int scratch = 2 * N;
        ClearRow(scratch);
        for (int i = 0; i < N; i++)
            if (a[ligne, 2 * N + i] != 0)
                RowSum(scratch, N + i);
        return R[scratch];
    }

    // La mesure aléatoire recopie l'ancien stabilisateur dans son partenaire.
    public int MeasureZ(int q)
    {
        // Un stabilisateur avec X sur q anticommute avec Zq : la mesure est aléatoire.
        int p = -1;
        for (int i = N; i < 2 * N; i++)
            if (X[i, q] != 0)
            {
                p = i;
                break;
            }

        // Sans tel pivot, la valeur est déterminée par le groupe stabilisateur.
        if (p < 0)
            return Deterministe(q);
        int result = rng.Next(2), scratch = 2 * N;
        // Conserver le pivot avant d’éliminer les autres lignes anticommutantes.
        CopyRow(scratch, p);
        for (int i = 0; i < 2 * N; i++)
            if (i != p && i != p - N && X[i, q] != 0)
                RowSum(i, scratch);
        CopyRow(p - N, scratch);
        ClearRow(p);
        // Le nouveau stabilisateur impose la valeur mesurée sur q.
        Z[p, q] = 1;
        R[p] = result;
        return result;
    }

    // Conjuguer par H transforme une mesure X en mesure Z.
    public int MeasureX(int q)
    {
        H(q);
        int m = MeasureZ(q);
        H(q);
        return m;
    }

    public void Apply(string gate, int q, int t = 0)
    {
        switch (gate)
        {
            case "h":
                H(q);
                break;
            case "s":
                S(q);
                break;
            case "x":
                XGate(q);
                break;
            case "z":
                ZGate(q);
                break;
            case "cnot":
                Cnot(q, t);
                break;
            case "t":
                throw new ArgumentException("T n'appartient pas à l'API Clifford");
            default:
                throw new ArgumentException("Porte inconnue");
        }
    }
}


In [6]:
// Témoin indépendant : le qubit zéro est le bit de poids fort du vecteur.
class StateVector
{
    public readonly int N;
    public readonly Complex[] V;
    readonly Random rng;
    public StateVector(int n, int seed = 0)
    {
        N = n;
        // Le vecteur complet contient une amplitude par état de base, soit 2^n cases.
        V = new Complex[1 << n];
        V[0] = 1;
        rng = new Random(seed);
    }

    public void H(int q)
    {
        int bit = 1 << (N - 1 - q);
        double h = 1 / Math.Sqrt(2);
        for (int i = 0; i < V.Length; i++)
            if ((i & bit) == 0)
            {
                // Traiter ensemble les amplitudes qui ne diffèrent que sur q.
                var a = V[i];
                var b = V[i | bit];
                V[i] = (a + b) * h;
                V[i | bit] = (a - b) * h;
            }
    }

    public void S(int q)
    {
        int bit = 1 << (N - 1 - q);
        for (int i = 0; i < V.Length; i++)
            if ((i & bit) != 0)
                V[i] *= Complex.ImaginaryOne;
    }

    public void T(int q)
    {
        int bit = 1 << (N - 1 - q);
        // La porte T multiplie uniquement le secteur où le bit q vaut 1.
        var phase = Complex.FromPolarCoordinates(1, Math.PI / 4);
        for (int i = 0; i < V.Length; i++)
            if ((i & bit) != 0)
                V[i] *= phase;
    }

    public void XGate(int q)
    {
        int bit = 1 << (N - 1 - q);
        for (int i = 0; i < V.Length; i++)
            if ((i & bit) == 0)
                (V[i], V[i | bit]) = (V[i | bit], V[i]);
    }

    public void ZGate(int q)
    {
        int bit = 1 << (N - 1 - q);
        for (int i = 0; i < V.Length; i++)
            if ((i & bit) != 0)
                V[i] = -V[i];
    }

    public void Cnot(int c, int t)
    {
        int cb = 1 << (N - 1 - c), tb = 1 << (N - 1 - t);
        for (int i = 0; i < V.Length; i++)
            if ((i & cb) != 0 && (i & tb) == 0)
                (V[i], V[i | tb]) = (V[i | tb], V[i]);
    }

    public void Apply(string gate, int q, int t = 0)
    {
        switch (gate)
        {
            case "h":
                H(q);
                break;
            case "s":
                S(q);
                break;
            case "x":
                XGate(q);
                break;
            case "z":
                ZGate(q);
                break;
            case "cnot":
                Cnot(q, t);
                break;
            case "t":
                T(q);
                break;
            default:
                throw new ArgumentException("Porte inconnue");
        }
    }

    public int MeasureZ(int q)
    {
        int bit = 1 << (N - 1 - q);
        // Additionner les modules carrés du secteur 1 donne sa probabilité.
        double p1 = 0;
        for (int i = 0; i < V.Length; i++)
            if ((i & bit) != 0)
                p1 += V[i].Magnitude * V[i].Magnitude;
        int m = rng.NextDouble() < p1 ? 1 : 0;
        // La division par racine de p renormalise l’état après projection.
        double p = m == 1 ? p1 : 1 - p1;
        for (int i = 0; i < V.Length; i++)
            V[i] = ((i & bit) != 0) == (m == 1) ? V[i] / Math.Sqrt(p) : Complex.Zero;
        return m;
    }

    public void Collapse(int q, int m)
    {
        int bit = 1 << (N - 1 - q);
        double p = 0;
        for (int i = 0; i < V.Length; i++)
            if (((i & bit) != 0) == (m == 1))
                p += V[i].Magnitude * V[i].Magnitude;
        // Un résultat de probabilité quasi nulle ne peut servir au témoin projeté.
        if (p < 1e-12)
            throw new Exception("Résultat impossible");
        for (int i = 0; i < V.Length; i++)
            V[i] = ((i & bit) != 0) == (m == 1) ? V[i] / Math.Sqrt(p) : Complex.Zero;
    }

    // Produit de Pauli appliqué directement à l'état : contrôle signé des stabilisateurs.
    public Complex Expectation(int[, ] x, int[, ] z, int row)
    {
        Complex s = Complex.Zero;
        for (int index = 0; index < V.Length; index++)
        {
            // cible suit les retournements X ; phase suit les signes Z et Y.
            int cible = index;
            Complex phase = Complex.One;
            for (int q = 0; q < N; q++)
            {
                int bit = 1 << (N - 1 - q), xv = x[row, q], zv = z[row, q];
                bool one = (index & bit) != 0;
                if (xv == 1 && zv == 1)
                    phase *= one ? -Complex.ImaginaryOne : Complex.ImaginaryOne;
                else if (zv == 1 && one)
                    phase = -phase;
                if (xv == 1)
                    cible ^= bit;
            }

            // Accumuler <état|P|état> vérifie le signe du stabilisateur.
            s += Complex.Conjugate(V[cible]) * phase * V[index];
        }

        return s;
    }
}


In [7]:
// Les règles calculées par matrices vérifient chaque cas de la conjugaison du tableau.
foreach (var entree in regles)
{
    var sim = new CHP(2, phases);
    int ligne = 2;
    sim.X[ligne, 0] = entree.Key.Item1 & 1;
    sim.Z[ligne, 0] = entree.Key.Item1 >> 1;
    sim.X[ligne, 1] = entree.Key.Item2 & 1;
    sim.Z[ligne, 1] = entree.Key.Item2 >> 1;
    sim.R[ligne] = 0;
    sim.Cnot(0, 1);
    int premier = sim.X[ligne, 0] + 2 * sim.Z[ligne, 0], second = sim.X[ligne, 1] + 2 * sim.Z[ligne, 1];
    if (premier != entree.Value.premier || second != entree.Value.second || (sim.R[ligne] == 0 ? 1 : -1) != entree.Value.signe)
        throw new Exception("Conjugaison CNOT du tableau incorrecte");
}

Console.WriteLine("Les 16 règles CNOT dérivées coïncident avec le tableau.");
record Gate(string Nom, int Q, int T = 0);
List<Gate> Circuit(int n, int depth, int seed, double fraction = 0)
{
    var r = new Random(seed * 7919 + n * 31 + depth + (int)(fraction * 1000));
    var gates = new List<Gate>();
    for (int i = 0; i < depth; i++)
    {
        if (r.NextDouble() < fraction)
        {
            gates.Add(new Gate("t", r.Next(n)));
            continue;
        }

        int k = r.Next(3), q = r.Next(n);
        if (k == 0)
            gates.Add(new Gate("h", q));
        else if (k == 1)
            gates.Add(new Gate("s", q));
        else
        {
            int t = r.Next(n - 1);
            if (t >= q)
                t++;
            gates.Add(new Gate("cnot", q, t));
        }
    }

    return gates;
}

CHP RunCHP(int n, List<Gate> gates, int seed)
{
    var s = new CHP(n, phases, seed);
    foreach (var g in gates)
        s.Apply(g.Nom, g.Q, g.T);
    return s;
}

StateVector RunSV(int n, List<Gate> gates, int seed)
{
    var s = new StateVector(n, seed);
    foreach (var g in gates)
        s.Apply(g.Nom, g.Q, g.T);
    return s;
}

string Stab(CHP s, int i)
{
    return (s.R[s.N + i] == 0 ? "+" : "-") + string.Concat(Enumerable.Range(0, s.N).Select(j => lettres[s.X[s.N + i, j] + 2 * s.Z[s.N + i, j]]));
}

var bell = RunCHP(2, new List<Gate> { new Gate("h", 0), new Gate("cnot", 0, 1) }, 0);
Console.WriteLine($"Bell : {Stab(bell, 0)}, {Stab(bell, 1)}");
var bellV = RunSV(2, new List<Gate> { new Gate("h", 0), new Gate("cnot", 0, 1) }, 0);
Console.WriteLine($"Poids Bell 00={bellV.V[0].Magnitude * bellV.V[0].Magnitude:F3}, 11={bellV.V[3].Magnitude * bellV.V[3].Magnitude:F3}");
int correles = 0;
for (int shot = 0; shot < 50; shot++)
{
    var s = RunCHP(2, new List<Gate> { new Gate("h", 0), new Gate("cnot", 0, 1) }, 1000 + shot);
    if (s.MeasureZ(0) == s.MeasureZ(1))
        correles++;
}

if (correles != 50)
    throw new Exception("Corrélation Bell incorrecte");
Console.WriteLine($"Bell : {correles}/50 mesures Z corrélées");
var ghz = new CHP(8, phases);
ghz.H(0);
for (int j = 1; j < 8; j++)
    ghz.Cnot(0, j);
Console.WriteLine("GHZ-8 : " + Stab(ghz, 0));
var mesures = Enumerable.Range(0, 8).Select(j => ghz.MeasureZ(j)).ToArray();
if (mesures.Distinct().Count() != 1)
    throw new Exception("GHZ incorrect");
Console.WriteLine("Mesures Z GHZ : " + string.Join(", ", mesures));


Les 16 règles CNOT dérivées coïncident avec le tableau.


Bell : +XX, +ZZ


Poids Bell 00=0.500, 11=0.500


Bell : 50/50 mesures Z corrélées


GHZ-8 : +XXXXXXXX


Mesures Z GHZ : 1, 1, 1, 1, 1, 1, 1, 1


**Lecture.** Le tableau lit Bell exactement comme les cours le décrivent :
stabilisateurs $+XX$ et $+ZZ$ — et la corrélation de mesure n'est pas énoncée, elle
**tombe** du formalisme : 50/50 tirs corrélés, aucune exception. GHZ tient dans les
mêmes lignes de bits : l'état le plus « intriqué » du zoo quantique coûte à l'arbitre
exactement ce que coûte $|0\rangle^{\otimes 8}$. C'est le premier résultat de la
leçon : *l'intrication n'est pas le mur* — la classe de circuits l'est.

## 4. La preuve croisée : l'arbitre ne ment pas

Un port écrit à la main doit prouver sa justesse, pas la réclamer. Le banc :

1. **valeurs propres exactes, zéro bruit** — pour des circuits Clifford aléatoires
   seedés, l'état final est un état stabilisateur : chacun de ses $n$ stabilisateurs
   $(-1)^{r_i} P_i$ vérifie $\langle P_i \rangle = (-1)^{r_i}$ **exactement**. Le CHP
   lit les $P_i, r_i$ dans son tableau ; l'état complet calcule $\langle P_i \rangle$
   par contraction tensorielle indépendante. Toute erreur de phase du port — le seul
   endroit où se tromper — inverse un signe et fait échouer le test brutalement.
2. **distributions de mesure échantillonnées** — les $n$ mesures $Z$ successives du
   CHP contre l'échantillonnage de la loi exacte $|\psi|^2$ : distance totale (TV)
   bornée par la marge de sondage (deux sondages de la même loi divergent de
   $\sim\sqrt{K/\pi}/\sqrt{\mathrm{shots}}$ sur $K$ sorties possibles).
3. **troisième juge** — avec [stim](https://github.com/quantumlib/Stim) installé dans le Python du pont,
   ses probabilités marginales de mesure donnent une seconde référence indépendante.

In [8]:
// Comparaison exacte des stabilisateurs signés à l'état complet pour les mêmes circuits.
int checks = 0;
foreach (int n in new[]
{
    4,
    6,
    8,
    10
}

)
    foreach (int seed in new[]
    {
        0,
        1,
        7
    }

    )
    {
        var gates = Circuit(n, 3 * n, seed);
        var a = RunCHP(n, gates, seed);
        var b = RunSV(n, gates, seed);
        for (int i = 0; i < n; i++)
        {
            // Chaque stabilisateur signé doit avoir la valeur propre imposée sur le vecteur.
            var e = b.Expectation(a.X, a.Z, n + i);
            if ((e - (a.R[n + i] == 0 ? 1 : -1)).Magnitude > 1e-9)
                throw new Exception("Stabilisateur incompatible avec état complet");
            checks++;
        }
    }

Console.WriteLine($"{checks} valeurs propres signées exactes à 1e-9");
// Deux sondages indépendants comparent les lois, sans utiliser le tableau comme oracle.
const int shots = 1500;
foreach (int n in new[]
{
    4,
    6
}

)
    foreach (int seed in new[]
    {
        0,
        7
    }

    )
    {
        var gates = Circuit(n, 3 * n, seed);
        int[] a = new int[1 << n], b = new int[1 << n];
        for (int shot = 0; shot < shots; shot++)
        {
            var c = RunCHP(n, gates, seed * 1009 + shot);
            int outcome = 0;
            for (int q = 0; q < n; q++)
                outcome = (outcome << 1) | c.MeasureZ(q);
            a[outcome]++;
        }

        var state = RunSV(n, gates, seed);
        var rng = new Random(2000 + seed + n);
        // La loi du vecteur donne un second échantillon indépendant du tableau.
        var cumul = new double[b.Length];
        double total = 0;
        for (int i = 0; i < b.Length; i++)
        {
            total += state.V[i].Magnitude * state.V[i].Magnitude;
            cumul[i] = total;
        }

        for (int shot = 0; shot < shots; shot++)
        {
            double u = rng.NextDouble() * total;
            int k = Array.BinarySearch(cumul, u);
            if (k < 0)
                k = ~k;
            b[Math.Min(k, b.Length - 1)]++;
        }

        double tv = 0;
        int bins = 0;
        for (int i = 0; i < a.Length; i++)
        {
            tv += Math.Abs(a[i] - b[i]) / (2.0 * shots);
            if (b[i] > 0)
                bins++;
        }

        // L’échelle de bruit dépend du nombre de cases observées et des tirs.
        double bruit = Math.Sqrt(bins / (Math.PI * shots));
        Console.WriteLine($"n={n}, seed={seed} : TV={tv:F3}, bruit~{bruit:F3}, {bins} sorties");
        if (tv >= Math.Max(0.12, 6 * bruit))
            throw new Exception("Distributions incompatibles");
    }

// Le pont cible uniquement Stim : le CHP et le vecteur restent implémentés en C#.
PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "stim");
PyObject stimScope;
using (Py.GIL())
{
    stimScope = Py.CreateScope();
    dynamic init = stimScope;
    init.Exec(@"import json, time, stim
version_stim = stim.__version__
def lancer(gates, n, seed):
    t0 = time.perf_counter()
    s = stim.TableauSimulator(seed=seed)
    for g in gates:
        if g['Nom'] == 'h': s.h(g['Q'])
        elif g['Nom'] == 's': s.s(g['Q'])
        elif g['Nom'] == 'cnot': s.cx(g['Q'], g['T'])
        else: raise ValueError('porte hors Clifford')
    duree = time.perf_counter() - t0
    return json.dumps({'seconds': duree, 'peek': [int(s.peek_z(i)) for i in range(n)]})");
    Console.WriteLine("Stim version " + init.Get<string>("version_stim"));
}

(double secondes, int[] peek) RunStim(int n, List<Gate> gates, int seed)
{
    if (gates.Any(g => g.Nom == "t"))
        throw new ArgumentException("T hors API Clifford de Stim");
    using (Py.GIL())
    {
        dynamic scope = stimScope;
        string j = JsonSerializer.Serialize(gates);
        scope.Exec($"reponse = lancer(json.loads({JsonSerializer.Serialize(j)}), {n}, {seed})");
        string result = scope.Get<string>("reponse");
        using var doc = JsonDocument.Parse(result);
        return (doc.RootElement.GetProperty("seconds").GetDouble(), doc.RootElement.GetProperty("peek").EnumerateArray().Select(x => x.GetInt32()).ToArray());
    }
}

int confirmations = 0;
foreach (int n in new[]
{
    6,
    8
}

)
    foreach (int seed in new[]
    {
        0,
        1,
        7
    }

    )
    {
        var gates = Circuit(n, 3 * n, seed);
        var juge = RunStim(n, gates, seed);
        var propre = RunCHP(n, gates, seed);
        for (int q = 0; q < n; q++)
        {
            // Un X dans une ligne stabilisatrice rend la mesure Z non déterministe.
            bool deterministic = true;
            for (int i = n; i < 2 * n; i++)
                if (propre.X[i, q] != 0)
                    deterministic = false;
            if ((juge.peek[q] != 0) != deterministic)
                throw new Exception("Stim et CHP divergent sur déterminisme");
            if (deterministic)
            {
                if (juge.peek[q] != 1 - 2 * propre.MeasureZ(q))
                    throw new Exception("Stim et CHP divergent sur valeur");
                confirmations++;
            }
        }
    }

Console.WriteLine($"Stim confirme {confirmations} mesures déterministes et tous les statuts déterministes/aléatoires.");


84 valeurs propres signées exactes à 1e-9


n=4, seed=0 : TV=0.003, bruit~0.021, 2 sorties


n=4, seed=7 : TV=0.001, bruit~0.021, 2 sorties


n=6, seed=0 : TV=0.029, bruit~0.058, 16 sorties


n=6, seed=7 : TV=0.024, bruit~0.041, 8 sorties


Stim version 1.15.0


Stim confirme 13 mesures déterministes et tous les statuts déterministes/aléatoires.


**Lecture.** Chaque erreur possible du port a un détecteur dédié : une erreur de
phase dans `rowsum` ou la règle du $\mathsf{CNOT}$ inverse une valeur propre ; une
erreur dans la mesure déplace la masse d'une distribution ; une erreur de convention
d'axes mélange les qubits et fait échouer les deux bancs à la fois. Le port que vous
venez de lire est passé **exact** sur les valeurs propres (aucun bruit qui puisse
cacher un signe) et **sous le seuil statistique** sur les distributions — le même
protocole qui, appliqué au brouillon de ce notebook, a intercepté une règle de phase
du $\mathsf{CNOT}$ fausse (le piège $X_c Z_t$ / $Y_c Y_t$ du §2) : le banc n'est pas
décoratif, il a déjà servi.

Dans ce rejeu C#, les quatre distances TV sont respectivement 0,003 et 0,001 pour $n=4$, puis 0,029 et 0,024 pour $n=6$ (seeds 0 et 7, 1 500 tirs) ; Stim confirme 13 valeurs déterministes et tous les statuts déterministes ou aléatoires. Les réalisations différent de celles de NumPy parce que les générateurs diffèrent. Ces valeurs sont des contrôles finis, pas une preuve générale de correction.


## 5. La mesure discriminante : le régime, pas l'objet

Protocole : les **mêmes circuits** (profondeur $3n$, portes Clifford uniformes, seeds
$(0, 1, 7)$) sur les trois machines, temps **médian** par taille. Puis la même
question côté suprématie : que se passe-t-il quand le circuit quitte la classe —
une fraction $f$ des portes remplacée par des $T$ ?

In [9]:
int[] nsChp =
{
    6,
    8,
    10,
    12,
    16,
    20,
    24,
    32,
    40,
    48,
    64
};
int[] nsSv =
{
    6,
    8,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    18,
    20,
    22
};
int[] nsStim =
{
    6,
    8,
    12,
    16,
    20,
    24,
    32,
    48,
    64,
    96,
    128,
    192,
    256
};
int[] seeds =
{
    0,
    1,
    7
};
Dictionary<int, (double median, double min, double max)> Banc(int[] tailles, Func<int, List<Gate>, int, double> mesurer)
{
    var sorties = new Dictionary<int, (double, double, double)>();
    foreach (int n in tailles)
    {
        var echantillons = new List<double>();
        foreach (int seed in seeds)
            echantillons.Add(mesurer(n, Circuit(n, 3 * n, seed), seed));
        echantillons.Sort();
        sorties[n] = (echantillons[1], echantillons[0], echantillons[2]);
    }

    return sorties;
}

double Temps(Action action)
{
    var montre = Stopwatch.StartNew();
    action();
    montre.Stop();
    return montre.Elapsed.TotalSeconds;
}

var chpTemps = Banc(nsChp, (n, g, s) => Temps(() => RunCHP(n, g, s)));
var svTemps = Banc(nsSv, (n, g, s) => Temps(() => RunSV(n, g, s)));
var stimTemps = Banc(nsStim, (n, g, s) => RunStim(n, g, s).secondes);
Console.WriteLine("Médianes et intervalles [min,max] sur trois seeds, secondes :");
foreach (int n in nsChp)
{
    var c = chpTemps[n];
    string v = svTemps.TryGetValue(n, out var x) ? $"{x.median:G4} [{x.min:G4},{x.max:G4}]" : "—";
    string st = stimTemps.TryGetValue(n, out var y) ? $"{y.median:G4} [{y.min:G4},{y.max:G4}]" : "—";
    Console.WriteLine($"n={n, 3} CHP={c.median:G4} [{c.min:G4},{c.max:G4}], état={v}, Stim={st}");
}

Console.WriteLine($"Ratio état/CHP à 20 : {svTemps[20].median / chpTemps[20].median:F3}");
Console.WriteLine($"Ratio CHP/Stim à 20 : {chpTemps[20].median / stimTemps[20].median:F3} (pont exclu)");
// Ajustement log2 t = a n + b et son coefficient de détermination.
double[] xx = nsSv.Select(n => (double)n).ToArray(), yy = nsSv.Select(n => Math.Log2(svTemps[n].median)).ToArray();
double mx = xx.Average(), my = yy.Average();
double pente = Enumerable.Range(0, xx.Length).Sum(i => (xx[i] - mx) * (yy[i] - my)) / xx.Sum(x => (x - mx) * (x - mx));
double ordonnee = my - pente * mx;
double r2 = 1 - Enumerable.Range(0, xx.Length).Sum(i => Math.Pow(yy[i] - pente * xx[i] - ordonnee, 2)) / yy.Sum(y => Math.Pow(y - my, 2));
var echelons = Enumerable.Range(0, nsSv.Length - 1).Select(i => Math.Pow(svTemps[nsSv[i + 1]].median / svTemps[nsSv[i]].median, 1.0 / (nsSv[i + 1] - nsSv[i]))).ToArray();
Console.WriteLine($"Pente={pente:F3} bits/qubit, R²={r2:F3}; échelons normalisés {echelons.Min():F3}..{echelons.Max():F3}");
foreach (int n in new[]
{
    30,
    40
}

)
    Console.WriteLine($"Extrapolation n={n} : {Math.Pow(2, pente * n + ordonnee):G4} s (non mesurée)");
var commun = nsSv.Where(n => chpTemps.ContainsKey(n)).ToArray();
var bascule = commun.Where(n => svTemps[n].median > chpTemps[n].median).Select(n => (int? )n).FirstOrDefault();
if (bascule is int nb)
{
    var s = svTemps[nb];
    var c = chpTemps[nb];
    double borneBasse = s.min / c.max, borneHaute = s.max / c.min;
    Console.WriteLine($"Première bascule observée n={nb}, ratio médian={s.median / c.median:F3} ; enveloppe [{borneBasse:F3}, {borneHaute:F3}]");
    if (borneBasse <= 1)
        Console.WriteLine("La borne basse franchit 1 : position sensible à la charge de la machine.");
}
else
{
    Console.WriteLine("Bascule non observée sur les tailles communes.");
    int? latent = commun.Where(n => svTemps[n].max / chpTemps[n].min > 1).Select(n => (int? )n).FirstOrDefault();
    if (latent is int n)
        Console.WriteLine($"La borne haute dépasse 1 dès n={n} : bascule latente.");
}


Médianes et intervalles [min,max] sur trois seeds, secondes :


n=  6 CHP=5.1E-06 [4.3E-06,4.71E-05], état=1.06E-05 [9.1E-06,5.51E-05], Stim=3.38E-05 [2.34E-05,4.41E-05]


n=  8 CHP=6.7E-06 [5.8E-06,8.5E-06], état=4.93E-05 [3.83E-05,5.49E-05], Stim=2.44E-05 [2.38E-05,2.7E-05]


n= 10 CHP=7.6E-06 [6.5E-06,7.9E-06], état=0.0002119 [0.0002063,0.0002891], Stim=—


n= 12 CHP=8.9E-06 [8.5E-06,9.5E-06], état=0.001137 [0.0009461,0.001211], Stim=4.1E-05 [3.53E-05,5.08E-05]


n= 16 CHP=1.34E-05 [1.33E-05,1.53E-05], état=0.02552 [0.02042,0.03229], Stim=4.41E-05 [4.3E-05,5.55E-05]


... (9 outputs left out)

Extrapolation n=30 : 955.1 s (non mesurée)


Extrapolation n=40 : 1.94E+06 s (non mesurée)


Première bascule observée n=6, ratio médian=2.078 ; enveloppe [0.193, 12.814]


La borne basse franchit 1 : position sensible à la charge de la machine.


In [10]:
Console.WriteLine("Les projections de la cellule précédente sont des extrapolations, non des mesures.");


Les projections de la cellule précédente sont des extrapolations, non des mesures.


In [11]:
// Comparer le même circuit sur les trois moteurs et signaler la frontière de leurs API.
Console.WriteLine("Une porte T sort de l'API Clifford ; elle peut cependant préserver un état stabilisateur.");
foreach (var spec in new (double fraction, bool une, string nom)[]
{
    (0.0, false, "Clifford pur"),
    (0.0, true, "une porte T"),
    (0.05, false, "5% de T"),
    (0.10, false, "10% de T")
}

)
{
    var gates = Circuit(8, 24, 0, spec.fraction);
    if (spec.une)
        gates.Insert(12, new Gate("t", 0));
    string c, s;
    try
    {
        RunCHP(8, gates, 0);
        c = "simulé";
    }
    catch (ArgumentException)
    {
        c = "refus T";
    }

    try
    {
        RunStim(8, gates, 0);
        s = "simulé";
    }
    catch (ArgumentException)
    {
        s = "pont/API refuse T";
    }

    double elapsed = Temps(() => RunSV(8, gates, 0));
    Console.WriteLine($"{spec.nom}, {gates.Count(g => g.Nom == "t")} porte(s) T réalisées : CHP {c}, Stim {s}, état complet {elapsed * 1000:F3} ms");
}


Une porte T sort de l'API Clifford ; elle peut cependant préserver un état stabilisateur.


Clifford pur, 0 porte(s) T réalisées : CHP simulé, Stim simulé, état complet 0.131 ms


une porte T, 1 porte(s) T réalisées : CHP refus T, Stim pont/API refuse T, état complet 0.181 ms


5% de T, 0 porte(s) T réalisées : CHP simulé, Stim simulé, état complet 0.055 ms


10% de T, 3 porte(s) T réalisées : CHP refus T, Stim pont/API refuse T, état complet 0.058 ms


**Lecture.** Trois constats, un seul message.

1. À circuits **Clifford purs**, la bascule est **mesurée**, pas extrapolée : l'état
   complet paie déjà plus que l'arbitre, et le ratio imprimé par le banc à
   $n = 20$ se lit dans la sortie committée de la cellule 22 et en donne l'ordre de grandeur. Le banc rend désormais chaque temps avec son
   **intervalle sur les trois seeds** ($[\min, \max]$), et le rapport des deux
   machines avec **sa propre enveloppe** ($\min$ et $\max$ des seeds, jamais la
   mediane du port prise pour un point) : ces chiffres sont des temps murs, et leur dispersion
   mesure la part de la charge de la machine dans la valeur affichée — d'où la
   prudence sur la *position* exacte de la bascule quand sa borne basse frôle
   $1\times$ (#19852). La pente globale de $\log_2 t$ par qubit et son $R^2$ sont affichés ; les échelons locaux
   sont normalisés par la différence de tailles, qui peut dépasser un qubit. La taille
   du vecteur double par qubit, mais les temps ne sont ni nécessairement monotones ni
   bornés inférieurement par un facteur deux à chaque mesure. Les projections à $n = 30$ et $n = 40$ sont **extrapolées du fit**
   ($R^2$ affiché), pas mesurées : niveau *Cité* de la hiérarchie du dépôt, étiqueté
   tel quel.
2. **Une seule porte $T$ ferme l'arbitre** — refus explicite, avec explication, pas un
   crash. Et l'outil du métier fait pareil : stim, le simulateur stabilisateur de
   référence, **n'a pas de méthode `t`** du tout et est parcouru effectivement pour les circuits Clifford purs. La frontière n'est pas un manque
   d'ingénierie : la porte sort de la classe Clifford. Gottesman–Knill garantit
   une simulation efficace dans cette classe ; son refus ne démontre pas la difficulté
   de tous les circuits avec $T$, ni que chaque état produit cesse d’être stabilisateur.
3. L'**état complet** accepte aussi $T$ : c'est une transformation supplémentaire
   du vecteur, dont le coût dépend du nombre et du type des opérations. Le tableau
   stabilisateur a une taille quadratique ; le coût du circuit dépend également du
   nombre de portes et de mesures.

> **Provenance de Stim.** Le port utilise réellement `stim` via Python.NET ; les circuits et résultats de comparaison traversent le pont, tandis que CHP et le vecteur sont natifs C#. Les temps Stim incluent la création du simulateur et son parcours, mais excluent la sérialisation et la traversée du pont. La version et le Python choisis sont visibles dans la sortie.

C'est la leçon de méthode, mesurée : une « suprématie quantique » est une assertion
conjointe sur **une classe de circuits et un régime de taille**. Réfuter proprement,
c'est exhiber le régime où l'assertion casse — ici, tout circuit Clifford, quelle que
soit la taille. Installer la claim, c'est montrer qu'on est sorti du régime — et $T$
est le billet d'entrée le plus faible du hors-classe.

## 6. Exercices

Les stubs s'exécutent sans erreur (règle C.1 du dépôt) ; les indices et étapes guident
sans écrire la solution. Les variables des bancs des sections précédentes restent en
portée.

### Exercice 1 — La phase par commutation, sans matrices

Le §1 dérive la table de phase en multipliant les matrices. Une autre voie, purement
algébrique, n'utilise que : (i) deux Pauli non triviales *distinctes* anti-commutent ;
(ii) l'identité $X \cdot Y = iZ$ (conséquence de $Y = iXZ$) ; (iii) l'hermiticité
($P_1 P_2 = (P_2 P_1)^\dagger$ inverse l'exposant). Écrire
`PhaseSansMatrices(p1, p2)` qui retourne $e$ tel que $P_1 P_2 = i^e P'$ **sans aucune
matrice**, puis la comparer à la table de phase dérivée case par case.

In [12]:
int? PhaseSansMatrices(string p1, string p2)
{
    // TODO : utiliser le cycle X→Y→Z et l'hermiticité, sans matrices.
    return null;
}

if (PhaseSansMatrices("X", "Y")is null)
    Console.WriteLine("Exercice 1 à compléter");
else
{
    foreach (string p in lettres)
        foreach (string q in lettres)
        {
            int? e = PhaseSansMatrices(p, q);
            int i = Array.IndexOf(lettres, p), j = Array.IndexOf(lettres, q);
            if (e is null || e != phases[i, j])
                throw new Exception($"Phase incorrecte pour {p}.{q}");
        }

    Console.WriteLine("16 phases concordent avec la dérivation matricielle");
}


Exercice 1 à compléter


### Exercice 2 — L'arbitre de classe

Écrire `VerdictSimulateur(n, portes)` : étant donné un circuit (liste de `Gate` comme
celles des bancs du §5) et une taille, retourner `"CHP"` si le circuit est Clifford
(l'arbitre polynomial s'applique), `"etat-complet"` s'il sort de la classe mais reste
dans les clous de l'état complet ($2^n \le 2^{22}$ amplitudes, la limite mémoire des
bancs de ce notebook), `"aucun"` sinon. La fonction ne **simule** rien : elle lit la
classe.

In [13]:
string VerdictSimulateur(int n, List<Gate> portes)
{
    // TODO : Clifford -> CHP ; avec T, état complet jusqu'à 22 qubits, sinon aucun.
    return null;
}

var dixPourcent8 = Circuit(8, 24, 0, 0.10);
var dixPourcent50 = Circuit(50, 150, 0, 0.10);
if (!dixPourcent8.Any(g => g.Nom == "t"))
    dixPourcent8.Add(new Gate("t", 0));
if (!dixPourcent50.Any(g => g.Nom == "t"))
    dixPourcent50.Add(new Gate("t", 0));
var casVerdict = new List<(int n, List<Gate> gates, string attendu)>
{
    (64, Circuit(64, 192, 0), "CHP"),
    (8, new List<Gate> { new Gate("h", 0), new Gate("t", 0) }, "etat-complet"),
    (22, new List<Gate> { new Gate("t", 0) }, "etat-complet"),
    (23, new List<Gate> { new Gate("t", 0) }, "aucun"),
    (8, dixPourcent8, "etat-complet"),
    (50, dixPourcent50, "aucun")
};
if (VerdictSimulateur(casVerdict[0].n, casVerdict[0].gates)is null)
    Console.WriteLine("Exercice 2 à compléter");
else
    foreach (var c in casVerdict)
    {
        var reponse = VerdictSimulateur(c.n, c.gates);
        if (reponse != c.attendu)
            throw new Exception($"Verdict incorrect pour n={c.n}");
        Console.WriteLine($"n={c.n}, portes={c.gates.Count} -> {reponse}");
    }


Exercice 2 à compléter


### Exercice 3 — Le mur, projeté par ses échelons

Le banc du §5 mesure les échelons locaux de l'état complet : le rapport
$t(n{+}1)/t(n)$, qubit par qubit. Recalculer la projection **à partir de ces échelons
seuls** : estimer le facteur multiplicatif par qubit (les petites tailles sont
dominées par les coûts fixes : ne garder que $n \ge 12$), sans supposer un plancher de temps universel, en déduire le temps projeté à $n = 30$, puis
le plus petit $n$ où la projection dépasse **une heure** par circuit.
Retourner `null` tant que l’exercice est ouvert ; une projection sans dépassement doit rendre un seuil `null`. La garde teste une série synthétique irrégulière et les données réellement mesurées.


In [14]:
(double facteur, double temps30, int? seuil)? Projection(int[] tailles, double[] temps)
{
    // TODO : médiane des rapports normalisés dès n=12, puis projection du dernier point.
    return null;
}

var taillesSynthetic = new[]
{
    12,
    14,
    16
};
var tempsSynthetic = new[]
{
    1.0,
    4.0,
    16.0
};
var essaiProjection = Projection(taillesSynthetic, tempsSynthetic);
if (essaiProjection is null)
    Console.WriteLine("Exercice 3 à compléter");
else
{
    // Une série irrégulière rend observable la division par l'écart de tailles.
    if (!double.IsFinite(essaiProjection.Value.facteur) || !double.IsFinite(essaiProjection.Value.temps30) || Math.Abs(essaiProjection.Value.facteur - 2) > 1e-10 || Math.Abs(essaiProjection.Value.temps30 - 262144) > 1e-6 || essaiProjection.Value.seuil != 24)
        throw new Exception("Projection synthétique incorrecte");
    var constant = Projection(taillesSynthetic, new[] { 1.0, 1.0, 1.0 });
    if (constant is null || !double.IsFinite(constant.Value.facteur) || !double.IsFinite(constant.Value.temps30) || Math.Abs(constant.Value.facteur - 1) > 1e-10 || Math.Abs(constant.Value.temps30 - 1) > 1e-10 || constant.Value.seuil != null)
        throw new Exception("Projection constante incorrecte");
    var reel = Projection(nsSv, nsSv.Select(n => svTemps[n].median).ToArray());
    if (reel is null || !double.IsFinite(reel.Value.facteur) || !double.IsFinite(reel.Value.temps30) || reel.Value.facteur <= 0)
        throw new Exception("Projection réelle absente ou non finie");
    var locaux = Enumerable.Range(0, nsSv.Length - 1).Where(i => nsSv[i] >= 12).Select(i => Math.Pow(svTemps[nsSv[i + 1]].median / svTemps[nsSv[i]].median, 1.0 / (nsSv[i + 1] - nsSv[i]))).OrderBy(x => x).ToArray();
    double f = locaux.Length % 2 == 1 ? locaux[locaux.Length / 2] : (locaux[locaux.Length / 2 - 1] + locaux[locaux.Length / 2]) / 2;
    double p30 = svTemps[nsSv.Last()].median * Math.Pow(f, 30 - nsSv.Last());
    int? seuil = nsSv.Last();
    double t = svTemps[nsSv.Last()].median;
    if (t <= 3600 && f <= 1)
        seuil = null;
    else
        while (t <= 3600)
        {
            t *= f;
            seuil++;
        }

    if (Math.Abs(reel.Value.facteur - f) > 1e-9 * Math.Max(1, f) || Math.Abs(reel.Value.temps30 - p30) > 1e-9 * Math.Max(1, p30) || reel.Value.seuil != seuil)
        throw new Exception("Projection réelle incorrecte");
    Console.WriteLine($"Facteur {f:F4}, n=30 {p30:G4} s, seuil {seuil?.ToString() ?? "absent"}");
}


Exercice 3 à compléter


## À retenir

| Geste | Où | Ce qui a été mesuré |
|---|---|---|
| Dériver, pas recopier | §1–§2 | Table de phase et conjugaison CHP **construites** par multiplication des matrices $2\times2$ ; le piège de phase $X_c Z_t / Y_c Y_t$ exhibé, pas affirmé |
| Construire l'organe | §3 | Port CHP complet (rowsum, portes, mesures — déterministe par lecture GF(2), aléatoire par re-stabilisation) ; Bell et GHZ lus dans le tableau, 50/50 corrélations mesurées |
| Prouver l'arbitre | §4 | **84 valeurs propres** stabilisatrices exactes au $10^{-9}$ (zéro bruit) ; distributions TV sous 6× le bruit de sondage ; Stim confirme les outcomes déterministes via Python.NET |
| Mesurer le régime | §5 | Bascule **mesurée**, pas extrapolée ; chaque temps rendu avec son **intervalle sur les seeds** et le ratio état-complet/port imprimé à $n = 20$ ; pente mesurée côté état complet ($R^2$ affiché) avec échelons locaux normalisés par différence de tailles ; projections $n = 30/40$ **extrapolées du fit, étiquetées telles** ; ces temps murs se lisent dans les sorties committées des cellules 21-22 et ne sont pas recopiés ici : un rafraîchissement de base ne fait plus dériver cette ligne |
| Fermer la classe | §5 | Une porte $T$ → refus du port et du pont vers l’API Clifford de Stim (pas de méthode `t`) ; l'état complet inchangé — la frontière est Clifford/le-reste, pas quantique/classique |

*Banc C# avec `stim` installé via Python.NET. Sur la même machine et le même code, des ré-exécutions complètes rendent un ratio qui varie — ce qui est stable, c'est le régime et son ordre de grandeur, pas le chiffre : seule la sortie committée fait foi. C'est pourquoi la ligne « Mesurer le régime » ne recopie pas les temps murs.*

**Ce que 2004 a fondé ici.** Aaronson et Gottesman n'ont pas « accéléré » la simulation
quantique : ils ont **délimité** la terre qui ne demandait pas de moteur quantique. La
leçon de méthode, mesurée sur ce banc, déborde le quantique : face à une claim
d'avantage — matériel, algorithmique, organisationnelle — la question qui réfute ou
installe n'est jamais « est-ce puissant ? » mais « **de quel régime parle-t-on, et où
casse-t-il ?** ». L'arbitre exécutable est la réponse la plus honnête qu'on puisse
rendre : il ne discute pas la claim, il tourne jusqu'à sa frontière.

**Hommage.** Le titre du notebook reprend le geste qu'Aaronson a le plus pratiqué —
déquantifier : chercher, derrière chaque « seul un quantique peut », le simulateur
classique du régime qui le contredit. Ce notebook est ce geste, retourné en banc
pédagogique : deux machines, une frontière, et chaque phase prouvée par construction.

Le Python versionné est la source primaire consultée pour ce port. Les références ci-dessous sont rapportées par ce carnet ; aucune lecture supplémentaire des PDF n’est revendiquée.

## Références

— Aaronson, S., & Gottesman, D. (2004). « Improved Simulation of Stabilizer Circuits ».
*Physical Review A* 70, 052328 (quant-ph/0406196).
— Gottesman, D. (1998). « The Heisenberg Representation of Quantum Computers » (quant-ph/9807006) — le théorème de Gottesman–Knill.
— Aaronson, S. (2007). « Are Quantum States Exponentially Long Vectors? ». *arXiv*
quant-ph/0701104 — le cadre de la déquantification et les classes de simulateurs.
— Gidney, C. (2021). « Stim: a fast stabilizer circuit simulator ». *Quantum* 5, 497.
— Nielsen, M. A., & Chuang, I. L. (2010). *Quantum Computation and Quantum
Information*, Cambridge University Press — chapitre 10.5, le formalisme stabilisateur.

## Vérifiez votre compréhension

1. Le banc du §4 rend **84 valeurs propres** stabilisatrices exactes à $10^{-9}$ près — zéro bruit — tandis que les distributions restent comparées sous un seuil statistique (TV sous $6\times$ le bruit de sondage). Pourquoi cette asymétrie de preuve entre les deux bancs ?
2. Une seule porte $T$ : l'arbitre refuse (avec explication), stim n'a pas de méthode `t`, et l'état complet, lui, ne change pas d'allure. Que garantit Gottesman–Knill, et que ne peut-on pas conclure du refus de $T$ ?
3. Le ratio état-complet/arbitre est **mesuré** à $n = 20$, chaque temps rendu avec son intervalle sur les trois seeds — et la ligne de conclusion ne recopie pas les temps murs. Pourquoi le banc rend-il des enveloppes plutôt qu'une médiane, et pourquoi s'interdire de recopier un chiffre dans la conclusion ?

<details><summary>Réponses</summary>

1. Les valeurs propres stabilisatrices sont des quantités **algébriques** : le tableau de Pauli les livre par lecture GF(2), sans tirage — l'exactitude au $10^{-9}$ n'est bruitée par rien. Les distributions de mesure, elles, exigent des tirages : la comparaison ne peut être que statistique, d'où le seuil en multiples du bruit de sondage. Deux natures de quantité, deux protocoles de preuve.
2. Gottesman-Knill dit : Clifford se simule en polynomial — il ne dit **pas** que le reste est dur. Le refus de $T$ exprime la limite de cette API, pas une borne de difficulté générale : $T|0\rangle=|0\rangle$ reste stabilisateur. Le coût de l’état complet dépend de sa taille et du nombre d’opérations exécutées.
3. Parce qu'un temps mur sur une machine chargée dit autant la charge que l'algorithme : l'intervalle des seeds mesure cette part, et la position exacte de la bascule reste prudente quand sa borne basse frôle $1\times$. Un chiffre recopié en conclusion dérive au premier rafraîchissement de base ; le renvoi aux sorties committées ne dérive jamais — c'est la discipline anti-dérive du dépôt.

</details>


## Pour aller plus loin

- **Carnet suivant** : [Complexity-06c — Classical at heart](Complexity-06c-ClassicalAtHeart-Python.ipynb) — la question miroir du 06 : non plus simuler un circuit, mais implémenter une opération quantique depuis un oracle classique.
- **Dans la même série** : [Complexity-06 — Simuler un circuit quantique classiquement](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) — la barrière $2^n$ du vecteur d'état, l'autre machine du banc ; [Complexity-05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) — l'arbitre $|\mathrm{perm}|^2$ et son mur mesuré.
- **Carnet voisin d'une autre série** : [Jauge-02 — De la symétrie globale à la jauge locale](../../Physique/Jauge/Jauge-02-Invariance-Jauge-U1-SU2-Python.ipynb) — la même question sous un autre angle : la liberté de la phase et la façon de la fixer ; le formalisme des stabilisateurs tranche par la convention de l'autovaleur +1, Jauge-02 par l'introduction d'un champ de jauge (Yang-Mills).
- **Lectures** : les cinq références (Aaronson-Gottesman 2004, Gottesman 1998, Aaronson 2007, Gidney 2021, Nielsen-Chuang ch. 10.5) figurent en fin de section « À retenir » ; aucune d'elles n'est archivée au gisement partagé à ce jour, et aucune ne porte les chiffres du carnet : ils sont produits par les bancs CHP et état complet natifs et par Stim à l’exécution.
